In [76]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

In [77]:
class Logistic_Regression_Model:
    def __init__(self, max_iter=10000, random_state=42):
        self.weight_matrix = 0
        self.bias_vector = 0
        self.max_iter = max_iter
        self.random_state = random_state
    
    def softmax(self,z):
        y = np.zeros(len(z))
        sum_z = np.sum(np.exp(z))
        for i,z_i in enumerate(z):
            y[i] = np.exp(z_i)/sum_z
        return y

    def cross_entropy(self,p,q):
        ce = 0
        for i in range(len(p)):
            ce += q[i]*np.log(p[i])
        ce = -ce
        return ce

    def fit(self, X_train, y_train, learning_rate):
        # Initialize weights to small random numbers and biases to zero
        num_samples = X_train.shape[0]
        num_features = X_train.shape[1]
        num_classes = len(np.unique(y_train))
        np_rng = np.random.default_rng(self.random_state)
        self.weight_matrix = np_rng.normal(loc=0.0, scale=0.01, size=(num_classes,num_features))
        self.bias_vector = np.zeros(num_classes)

        # One hot encoding for classes
        y_train_e = np.zeros((num_samples,num_classes))
        y_train_e[np.arange(num_samples), y_train] = 1

        yhat_train = np.zeros((num_samples,num_classes))
        previous_loss = -1
        flag_end = False
        num_iter = 0
        # Train until loss has small change or max number of iterations is reached
        while((not flag_end) and (num_iter<=self.max_iter)):
            num_iter += 1
            loss_sum = 0

            # Compute z, yhat, and cross entropy for each sample
            for i,x in enumerate(X_train):
                z = np.add(np.matmul(self.weight_matrix,x),self.bias_vector)
                yhat_train[i] = self.softmax(z)
                loss_sum += self.cross_entropy(yhat_train[i],y_train_e[i])

            # Average loss over the samples
            loss = loss_sum/num_samples

            # End training if loss is barely changing (converged)
            if(previous_loss != -1):
                if(abs(loss - previous_loss) <= 0.000001):
                    flag_end = True
            
            previous_loss = loss

            # Gradient calculations for W and b
            w_grad = (1/num_samples)*((yhat_train-y_train_e).T@X_train)
            b_grad = (1/num_samples)*(np.sum(yhat_train-y_train_e, axis=0))

            # Update weight matrix and bias based on gradient and learning rate
            self.weight_matrix -= learning_rate*w_grad
            self.bias_vector -= learning_rate*b_grad

    def predict(self, X_test, y_test):
        num_samples = X_test.shape[0]
        num_classes = len(np.unique(y_test))

        # One hot encoding
        y_test_e = np.zeros((num_samples,num_classes))
        y_test_e[np.arange(num_samples), y_test] = 1

        yhat_test = np.zeros((num_samples,num_classes))
        # Compute z, and yhat for each sample
        for i,x in enumerate(X_test):
                z = np.add(np.matmul(self.weight_matrix,x),self.bias_vector)
                yhat_test[i] = self.softmax(z)

        # Choose most likely class (highest probability) for each sample
        yhat_test_class = np.argmax(yhat_test, axis=1)
        return yhat_test_class


In [78]:
RANDOM_STATE = 42

features = pd.read_csv("../data/features.txt", sep=r"\s+", header=None, names=["index", "feature_name"])

X_train = pd.read_csv("../data/train/X_train.txt", sep=r"\s+", header=None)
y_train = pd.read_csv("../data/train/y_train.txt", sep=r"\s+", header=None, names=["activity"])
subjects_train = pd.read_csv("../data/train/subject_train.txt", sep=r"\s+", header=None, names=["subject"])

X_test = pd.read_csv("../data/test/X_test.txt", sep=r"\s+", header=None)
y_test = pd.read_csv("../data/test/y_test.txt", sep=r"\s+", header=None, names=["activity"])
subjects_test = pd.read_csv("../data/test/subject_test.txt", sep=r"\s+", header=None, names=["subject"])

activity_labels = pd.read_csv("../data/activity_labels.txt", sep=r"\s+", header=None, names=["index", "activity_name"])

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (7352, 561) | X_test: (2947, 561)


In [79]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train["activity"].to_numpy())
y_test_enc = le.transform(y_test["activity"].to_numpy())
target_names = (
    activity_labels.set_index("index").loc[le.classes_, "activity_name"].astype(str).tolist()
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [80]:
model = Logistic_Regression_Model(random_state=RANDOM_STATE)
model.fit(X_train_s, y_train_enc, 0.05)
pred = model.predict(X_test_s, y_test_enc)
rows = []
rows.append(
            {
                "test_accuracy": accuracy_score(y_test_enc, pred),
                "test_macro_precision": precision_score(
                    y_test_enc, pred, average="macro", zero_division=0
                ),
                "test_macro_recall": recall_score(
                    y_test_enc, pred, average="macro", zero_division=0
                ),
                "test_macro_f1": f1_score(
                    y_test_enc, pred, average="macro", zero_division=0
                ),
            }
        )

results = pd.DataFrame(rows)
display(results)

print(classification_report(y_test_enc, pred, target_names=target_names, digits=3))

,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1
0,0.947404,0.950939,0.946722,0.94785


                    precision    recall  f1-score   support

           WALKING      0.941     0.992     0.966       496
  WALKING_UPSTAIRS      0.957     0.941     0.949       471
WALKING_DOWNSTAIRS      0.985     0.948     0.966       420
           SITTING      0.955     0.868     0.909       491
          STANDING      0.868     0.962     0.913       532
            LAYING      1.000     0.970     0.985       537

          accuracy                          0.947      2947
         macro avg      0.951     0.947     0.948      2947
      weighted avg      0.950     0.947     0.948      2947

